# Metis · Hidden Market Regimes

Can market states discovered without event labels help describe risk? This notebook runs a complete chronological experiment. **Default data is synthetic: do not interpret its returns as market evidence.**

## 1. Setup and reproducibility
Kaggle normally provides NumPy, pandas, SciPy, scikit-learn, matplotlib and PyYAML. For downloads, install yfinance if needed with internet enabled. A CSV works offline. The source is embedded below, so no repository clone is required.

In [ ]:
from pathlib import Path
import json, sys
PROJECT = Path.cwd() / "metis_notebook_project"
PROJECT.mkdir(exist_ok=True)
SOURCES = {'src/__init__.py': '"""Metis: reproducible market regime research."""\n', 'src/backtest.py': '"""Close-to-close execution, full L1 traded notional, drift-aware turnover."""\nimport numpy as np\nimport pandas as pd\n\ndef run_backtest(prices, equity_targets, cost=.001):\n    if not 0 <= cost < .1:\n        raise ValueError(\'Invalid cost rate\')\n    targets = equity_targets.reindex(prices.index)\n    if targets.isna().any() or not targets.between(0,1).all():\n        raise ValueError(\'Targets must cover every date and lie in [0,1]\')\n    returns = prices[[\'equity\',\'bonds\']].pct_change(fill_method=None).fillna(0)\n    held = np.array([0., 0.]); cash = 1.\n    records = []\n    for i, date in enumerate(prices.index):\n        # Decision at close t-1, execution at close t, first earning return t+1.\n        gross = float(held @ returns.iloc[i].to_numpy())\n        drift = held*(1+returns.iloc[i].to_numpy())/(1+gross)\n        drift_cash = cash/(1+gross)\n        if i == 0:\n            target = drift\n            turnover = 0.\n        else:\n            eq = float(targets.iloc[i-1]); target = np.array([eq, 1-eq])\n            # Exact post-fee NAV target: solve fee = c * |target*(NAV-fee)-holdings|.\n            fee = 0.\n            for _ in range(30):\n                fee = cost*np.abs(target*(1-fee)-drift).sum()\n            turnover = float(np.abs(target*(1-fee)-drift).sum())\n            drift_cash = 0.\n        fee = cost*turnover\n        net = (1+gross)*(1-fee)-1\n        held, cash = target, drift_cash\n        records.append((gross, net, turnover, fee, held[0]))\n    return pd.DataFrame(records, index=prices.index, columns=[\'gross\',\'net\',\'turnover\',\'cost_fraction\',\'equity_weight\'])\n', 'src/data_loader.py': '"""CSV adapter, optional Yahoo download, and explicitly synthetic smoke data."""\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nTICKERS = {\'equity\':\'SPY\', \'bonds\':\'IEF\', \'gold\':\'GLD\', \'oil\':\'USO\', \'dollar\':\'UUP\', \'vix\':\'^VIX\'}\n\ndef load_csv(path):\n    d = pd.read_csv(path, parse_dates=[\'date\']).set_index(\'date\').sort_index()\n    if d.index.has_duplicates:\n        raise ValueError(\'Duplicate dates are not permitted\')\n    required = [\'equity\', \'bonds\']\n    if not set(required).issubset(d.columns):\n        raise ValueError(\'CSV requires date, equity, bonds; adjusted positive prices\')\n    if d[required].isna().any().any() or (d[required] <= 0).any().any():\n        raise ValueError(\'Equity/bond prices must be complete and positive; no price filling\')\n    if not np.isfinite(d.to_numpy(dtype=float)).all() or (d <= 0).any().any():\n        raise ValueError(\'Use complete, finite, positive observations; inspect missing sessions first\')\n    return d\n\ndef download(path, start=\'2005-01-01\', end=None):\n    import yfinance as yf\n    series = {}\n    for name, ticker in TICKERS.items():\n        raw = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)\n        if raw.empty:\n            raise ValueError(f\'No data returned for {ticker}\')\n        series[name] = raw[\'Close\'].squeeze().rename(name)\n    d = pd.concat(series.values(), axis=1).dropna()\n    d.index.name = \'date\'\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    d.to_csv(path)\n    Path(str(path)+\'.source.txt\').write_text(\'Yahoo Finance via yfinance; adjusted Close; ETFs \'+str(TICKERS)+\'\\nCommon complete sessions; inspect inception dates and data licensing before redistribution.\\n\')\n    return d\n\ndef synthetic(n=1800, seed=42):\n    """Software demonstration only. Not evidence about actual markets."""\n    rng = np.random.default_rng(seed)\n    states = np.zeros(n, int)\n    for i in range(1, n):\n        states[i] = states[i-1] if rng.random()<.975 else rng.integers(0, 3)\n    vol = np.array([.006, .014, .03])[states]\n    r = rng.normal(0, 1, (n, 5))*vol[:, None]*np.array([1, .3, .65, 1.3, .25])\n    r[:, 0] += np.array([.0005, 0, -.001])[states]\n    d = pd.DataFrame(100*np.exp(np.cumsum(r, axis=0)), columns=[\'equity\',\'bonds\',\'gold\',\'oil\',\'dollar\'], index=pd.bdate_range(\'2015-01-01\', periods=n))\n    d[\'vix\'] = 10 + 650*vol + rng.uniform(0, 3, n)\n    d.index.name = \'date\'\n    return d\n', 'src/features.py': "import numpy as np\nimport pandas as pd\n\ndef make_features(prices, windows=(5,20,60)):\n    r = prices.pct_change(fill_method=None)\n    x = pd.DataFrame(index=prices.index)\n    x['return_1'] = r.equity\n    for w in windows:\n        # Cumulative returns already express momentum; avoid duplicate columns.\n        x[f'return_{w}'] = prices.equity.pct_change(w, fill_method=None)\n        x[f'vol_{w}'] = r.equity.rolling(w).std()*np.sqrt(252)\n    x['vol_ratio'] = x[f'vol_{windows[0]}']/x[f'vol_{windows[-1]}']\n    x['drawdown'] = prices.equity/prices.equity.rolling(252, min_periods=60).max()-1\n    for c in prices.columns:\n        if c != 'equity':\n            x[f'{c}_change'] = r[c]\n        if c in ('bonds','gold','oil'):\n            x[f'corr_{c}'] = r.equity.rolling(60).corr(r[c])\n    if 'vix' in prices:\n        x['vix_level'] = prices.vix\n    return x.replace([np.inf,-np.inf], np.nan).dropna()\n", 'src/hmm_model.py': '"""Diagonal Gaussian HMM: EM training and strictly causal forward filtering."""\nimport numpy as np\nfrom scipy.special import logsumexp\nfrom sklearn.cluster import KMeans\n\nclass GaussianHMM:\n    def __init__(self, n_components=4, seed=42, n_iter=100, tol=1e-4):\n        self.k, self.seed, self.n_iter, self.tol = n_components, seed, n_iter, tol\n\n    def emissions(self, x):\n        return -.5 * (np.log(2*np.pi*self.var).sum(1)[None, :] +\n                      ((x[:, None, :]-self.means)**2/self.var).sum(2))\n\n    def forward(self, e, initial=None):\n        a = np.empty_like(e)\n        a[0] = np.log(self.start if initial is None else initial) + e[0]\n        for t in range(1, len(e)):\n            a[t] = e[t] + logsumexp(a[t-1, :, None] + np.log(self.transition), axis=0)\n        return a\n\n    def fit(self, x):\n        x = np.asarray(x, float)\n        labels = KMeans(self.k, random_state=self.seed, n_init=10).fit_predict(x)\n        self.means = np.array([x[labels == j].mean(0) for j in range(self.k)])\n        self.var = np.array([x[labels == j].var(0)+.05 for j in range(self.k)])\n        self.start = np.full(self.k, 1/self.k)\n        self.transition = .90*np.eye(self.k)+.10/self.k\n        self.history = []\n        for _ in range(self.n_iter):\n            e = self.emissions(x)\n            a = self.forward(e)\n            ll = logsumexp(a[-1])\n            b = np.zeros_like(e)\n            lt = np.log(self.transition)\n            for t in range(len(x)-2, -1, -1):\n                b[t] = logsumexp(lt + e[t+1][None, :] + b[t+1][None, :], axis=1)\n            gamma = np.exp(a+b-ll)\n            counts = np.zeros((self.k, self.k))\n            for t in range(len(x)-1):\n                counts += np.exp(a[t, :, None]+lt+e[t+1][None, :]+b[t+1][None, :]-ll)\n            self.start = np.maximum(gamma[0], 1e-8); self.start /= self.start.sum()\n            self.transition = counts + 1e-6\n            self.transition /= self.transition.sum(1, keepdims=True)\n            mass = gamma.sum(0)[:, None] + 1e-12\n            self.means = gamma.T @ x / mass\n            self.var = np.maximum(gamma.T @ (x*x)/mass-self.means**2, .01)\n            self.history.append(float(ll))\n            if len(self.history)>1 and abs(ll-self.history[-2]) < self.tol*(1+abs(self.history[-2])):\n                break\n        return self\n\n    def filter(self, x, previous=None):\n        """P(S_t | X_1,...,X_t); never backward-smoothed or Viterbi labels."""\n        initial = None if previous is None else previous @ self.transition\n        a = self.forward(self.emissions(np.asarray(x)), initial)\n        return np.exp(a-logsumexp(a, axis=1, keepdims=True))\n\n    def score(self, x, previous=None):\n        initial = None if previous is None else previous @ self.transition\n        return float(logsumexp(self.forward(self.emissions(np.asarray(x)), initial)[-1]))\n', 'src/metrics.py': "import numpy as np\n\ndef summarize(bt):\n    r = bt.net\n    nav = (1+r).cumprod()\n    annual = nav.iloc[-1]**(252/len(r))-1\n    vol = r.std()*np.sqrt(252)\n    downside = np.sqrt(np.mean(np.minimum(r,0)**2))*np.sqrt(252)\n    dd = (nav/nav.cummax().clip(lower=1)-1).min()\n    return {'annualized_return':annual, 'annualized_volatility':vol,\n            'sharpe_rf_zero':r.mean()*252/vol if vol else np.nan,\n            'sortino_target_zero':r.mean()*252/downside if downside else np.nan,\n            'max_drawdown':dd, 'calmar':annual/abs(dd) if dd else np.nan,\n            'annual_turnover':bt.turnover.mean()*252,\n            'sum_cost_fractions':bt.cost_fraction.sum(), 'ending_nav':nav.iloc[-1]}\n", 'src/pipeline.py': 'import json\nimport hashlib\nfrom pathlib import Path\nimport pickle\nimport numpy as np\nimport pandas as pd\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.decomposition import PCA\nfrom .features import make_features\nfrom .regimes import compare\nfrom .hmm_model import GaussianHMM\nfrom .backtest import run_backtest\nfrom .metrics import summarize\nfrom .visualization import plots\n\ndef run(prices, cfg, out=\'outputs\', demo=False):\n    out=Path(out); (out/\'results\').mkdir(parents=True,exist_ok=True); (out/\'models\').mkdir(exist_ok=True)\n    f=make_features(prices, cfg.get(\'windows\',[5,20,60]))\n    if demo:\n        a,b=int(len(f)*.6),int(len(f)*.8)\n    else:\n        a=int((f.index<=cfg[\'train_end\']).sum()); b=int((f.index<=cfg[\'validation_end\']).sum())\n    if min(a,b-a,len(f)-b)<100:\n        raise ValueError(\'Need at least 100 complete observations in each chronological split\')\n    scaler=StandardScaler().fit(f.iloc[:a]); x=scaler.transform(f)\n    comparison,model,all_models=compare(x[:a],x[a:b],cfg[\'states\'],cfg[\'seed\'])\n    comparison.to_csv(out/\'results/model_comparison.csv\',index=False)\n    probs=model.filter(x)\n    for (kind,k), fitted in all_models.items():\n        if kind == "gmm":\n            pd.DataFrame(fitted.predict_proba(x),index=f.index).to_csv(out/f"results/gmm_{k}_probabilities.csv")\n    # State naming/risk order is learned from training data ONLY.\n    train_labels=probs[:a].argmax(1)\n    profiles=f.iloc[:a].groupby(train_labels).mean().reindex(range(model.k))\n    vol_col=f"vol_{cfg.get(\'windows\',[5,20,60])[1]}"\n    fallback=pd.Series(scaler.inverse_transform(model.means)[:,f.columns.get_loc(vol_col)],index=range(model.k))\n    risk=profiles[vol_col].fillna(fallback).sort_values().index\n    weights=np.empty(model.k); weights[risk]=np.interp(np.linspace(0,1,model.k),np.linspace(0,1,len(cfg[\'risk_weights\'])),cfg[\'risk_weights\'])\n    profiles[\'risk_rank\']=pd.Series(np.argsort(np.argsort(profiles[vol_col].fillna(fallback))),index=profiles.index)\n    profiles.to_csv(out/\'results/training_state_profiles.csv\')\n    pd.DataFrame(model.transition).to_csv(out/\'results/transition_matrix.csv\',index=False)\n    pd.DataFrame(probs,index=f.index,columns=[f\'state_{i}\' for i in range(model.k)]).to_csv(out/\'results/filtered_probabilities.csv\')\n    test=f.index[b:]; p=prices.loc[test]; target=pd.Series(probs[b:]@weights,index=test)\n    portfolios={\'buy_hold\':run_backtest(p,pd.Series(1.,index=test),cfg[\'transaction_cost\']),\n                \'fixed_60_40\':run_backtest(p,pd.Series(.6,index=test),cfg[\'transaction_cost\']),\n                \'regime\':run_backtest(p,target,cfg[\'transaction_cost\'])}\n    metrics=pd.DataFrame({name:summarize(bt) for name,bt in portfolios.items()}).T\n    metrics.to_csv(out/\'results/portfolio_metrics.csv\')\n    for name,bt in portfolios.items(): bt.to_csv(out/f\'results/{name}_backtest.csv\')\n    sensitivity=[]\n    for cost in [0,.0005,.001,.002,.005]:\n        for scale in [.8,1.,1.2]:\n            result=summarize(run_backtest(p,(target*scale).clip(0,1),cost))\n            sensitivity.append({\'cost\':cost,\'allocation_scale\':scale,**result})\n    pd.DataFrame(sensitivity).to_csv(out/\'results/cost_allocation_sensitivity.csv\',index=False)\n    # Validation-only structural robustness: do not select on test outcomes.\n    structural=[]\n    for window_set in [[3,15,45],[5,20,60],[10,30,90]]:\n        alt=make_features(prices,window_set)\n        for trim in [0,.15]:\n            train=alt.loc[alt.index<=f.index[a-1]]; train=train.iloc[int(len(train)*trim):]\n            val=alt.loc[(alt.index>f.index[a-1]) & (alt.index<=f.index[b-1])]\n            sc=StandardScaler().fit(train); tr=sc.transform(train); va=sc.transform(val)\n            hm=GaussianHMM(model.k,cfg[\'seed\'],n_iter=60).fit(tr)\n            pr=hm.filter(va,hm.filter(tr)[-1])\n            structural.append({\'windows\':str(window_set),\'training_trim\':trim,\'validation_loglik_per_day\':hm.score(va,hm.filter(tr)[-1])/len(va),\'switch_fraction\':float(np.mean(np.diff(pr.argmax(1))!=0))})\n    pd.DataFrame(structural).to_csv(out/\'results/window_training_sensitivity.csv\',index=False)\n    state=probs.argmax(1); starts=np.r_[0,np.flatnonzero(np.diff(state))+1]; ends=np.r_[starts[1:],len(state)]\n    pd.DataFrame({\'state\':state[starts],\'start\':f.index[starts],\'duration\':ends-starts,\'boundary_censored\':(starts==0)|(ends==len(state))}).to_csv(out/\'results/durations.csv\',index=False)\n    entropy=-(probs*np.log(np.maximum(probs,1e-15))).sum(1)\n    pd.DataFrame({\'entropy\':entropy,\'confidence\':probs.max(1)},index=f.index).to_csv(out/\'results/uncertainty.csv\')\n    perstate=[]\n    for name,bt in portfolios.items():\n        # State known for the decision underlying that day\'s earned return.\n        decision=pd.Series(state[b:],index=test).shift(2)\n        for s,g in bt.groupby(decision):\n            perstate.append({\'portfolio\':name,\'decision_state\':int(s),\'sessions\':len(g),\'mean_daily_net\':g.net.mean(),\'daily_volatility\':g.net.std()})\n    pd.DataFrame(perstate).to_csv(out/\'results/performance_by_state.csv\',index=False)\n    title=\'SYNTHETIC DEMO — no financial conclusions\' if demo else \'Historical research — filtered states; frozen training fit\'\n    plots(prices,f,probs,model,profiles.drop(columns=\'risk_rank\'),PCA(2).fit(x[:a]).transform(x),portfolios,out/\'figures\',title)\n    with open(out/\'models/fitted.pkl\',\'wb\') as stream: pickle.dump({\'model\':model,\'scaler\':scaler,\'features\':list(f),\'weights\':weights},stream)\n    manifest={\'input_sha256\':hashlib.sha256(prices.to_csv().encode()).hexdigest(),\'synthetic\':demo,\'selected_states\':model.k,\'train_end\':str(f.index[a-1]),\'validation_end\':str(f.index[b-1]),\'test_start\':str(test[0]),\'test_end\':str(test[-1]),\'rows\':len(f),\'config\':cfg,\'hmm_iterations\':len(model.history),\'hmm_loglik_history\':model.history}\n    (out/\'results/run_manifest.json\').write_text(json.dumps(manifest,indent=2))\n    (out/\'results/REPORT.md\').write_text(\'# Metis run\\n\\n\'+title+\'\\n\\n```\\n\'+metrics.to_string()+\'\\n```\\n\\nModel selection uses validation likelihood; test is untouched until evaluation. Sensitivities are exploratory, not independent confirmation. Labels describe training statistics, not confirmed economic regimes.\\n\')\n    return metrics\n', 'src/regimes.py': "import numpy as np\nimport pandas as pd\nfrom sklearn.cluster import KMeans\nfrom sklearn.mixture import GaussianMixture\nfrom sklearn.metrics import silhouette_score, adjusted_rand_score\nfrom .hmm_model import GaussianHMM\n\ndef compare(train, validation, ks, seed):\n    rows, models = [], {}\n    for k in ks:\n        for kind in ('kmeans','gmm','hmm'):\n            if kind == 'kmeans':\n                m = KMeans(k, n_init=10, random_state=seed).fit(train)\n                label = m.predict(validation)\n                other = KMeans(k, n_init=10, random_state=seed+1).fit(train)\n                score = m.score(validation)/len(validation)\n                stability = adjusted_rand_score(label, other.predict(validation))\n            elif kind == 'gmm':\n                m = GaussianMixture(k, covariance_type='diag', n_init=3, reg_covar=.01, random_state=seed).fit(train)\n                label = m.predict(validation); score = m.score(validation); stability = np.nan\n            else:\n                m = GaussianHMM(k, seed).fit(train)\n                previous = m.filter(train)[-1]\n                label = m.filter(validation, previous).argmax(1)\n                score = m.score(validation, previous)/len(validation); stability = np.nan\n            silhouette = silhouette_score(validation, label, sample_size=min(1000,len(label)), random_state=seed) if 1<len(set(label))<len(label) else np.nan\n            rows.append({'model':kind,'k':k,'validation_score':score,'silhouette':silhouette,'seed_stability_ari':stability})\n            models[kind,k] = m\n    table = pd.DataFrame(rows)\n    # Select HMM on validation likelihood only; never on test portfolio return.\n    winner = table[table.model=='hmm'].sort_values('validation_score', ascending=False).iloc[0]\n    return table, models['hmm',int(winner.k)], models\n", 'src/visualization.py': "from pathlib import Path\nfrom matplotlib.patches import Patch\nfrom matplotlib.colors import ListedColormap, BoundaryNorm\nimport matplotlib\nmatplotlib.use('Agg')\nimport matplotlib.pyplot as plt\nimport numpy as np\n\ndef plots(prices, features, probs, model, profiles, pca, portfolios, out, title):\n    out = Path(out); out.mkdir(parents=True,exist_ok=True)\n    plt.rcParams.update({'figure.figsize':(12,5),'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,'grid.alpha':.15})\n    labels = probs.argmax(1); dates=features.index\n    colors = plt.get_cmap('tab10')\n    cmap=ListedColormap([colors(i) for i in range(model.k)])\n    norm=BoundaryNorm(np.arange(model.k+1)-.5,model.k)\n    def save(name):\n        plt.suptitle(title, fontsize=9, color='gray'); plt.tight_layout(); plt.savefig(out/f'{name}.png',dpi=140); plt.close()\n    def shade(ax):\n        starts = np.r_[0,np.flatnonzero(np.diff(labels))+1]; ends=np.r_[starts[1:],len(labels)-1]\n        for a,b in zip(starts,ends): ax.axvspan(dates[a],dates[b],color=colors(labels[a]),alpha=.17)\n    fig,ax=plt.subplots(); ax.plot(dates,prices.loc[dates,'equity'],color='#172d47'); shade(ax); ax.legend(handles=[Patch(facecolor=colors(i),alpha=.4,label=f'State {i}') for i in range(model.k)],ncol=model.k,loc='upper left'); ax.set_title('Equity price · causal filtered states'); save('01_market_regimes')\n    plt.figure(); plt.scatter(dates,labels,c=labels,cmap=cmap,norm=norm,s=3); plt.ylabel('State'); save('02_timeline')\n    plt.figure(); z=(profiles-profiles.mean())/profiles.std().replace(0,1); plt.imshow(z,aspect='auto',cmap='RdBu_r'); plt.xticks(range(len(z.columns)),z.columns,rotation=90); plt.yticks(range(len(z)),z.index); plt.colorbar(); save('03_profiles')\n    plt.figure(); plt.imshow(model.transition,vmin=0,vmax=1,cmap='Blues'); plt.xlabel('To'); plt.ylabel('From'); plt.colorbar(); save('04_transition')\n    plt.figure(); plt.stackplot(dates,probs.T,labels=[f'State {i}' for i in range(probs.shape[1])]); plt.legend(loc='upper left'); save('05_probabilities')\n    plt.figure(); plt.scatter(pca[:,0],pca[:,1],c=labels,cmap=cmap,norm=norm,s=4,alpha=.5); plt.xlabel('PC1'); plt.ylabel('PC2'); save('06_pca')\n    for number,kind in [(7,'drawdown'),(8,'equity'),(9,'sharpe'),(10,'volatility')]:\n        plt.figure()\n        for name,bt in portfolios.items():\n            r=bt.net; nav=(1+r).cumprod()\n            y={'drawdown':nav/nav.cummax().clip(lower=1)-1,'equity':nav,'sharpe':r.rolling(126).mean()/r.rolling(126).std()*np.sqrt(252),'volatility':r.rolling(126).std()*np.sqrt(252)}[kind]\n            plt.plot(y,label=name)\n        plt.title('Held-out test · '+kind); plt.legend(); save(f'{number:02d}_{kind}')\n    starts=np.r_[0,np.flatnonzero(np.diff(labels))+1]; durations=np.diff(np.r_[starts,len(labels)])\n    plt.figure(); plt.hist(durations,bins=30); plt.xlabel('Observed run length (sessions; boundary runs censored)'); save('11_durations')\n    plt.figure(); profiles['return_1'].plot.bar(); plt.ylabel('Contemporaneous mean daily return (descriptive)'); save('12_returns_by_state')\n    plt.figure(); plt.imshow(features.corr(),vmin=-1,vmax=1,cmap='RdBu_r'); plt.colorbar(); plt.xticks(range(len(features.columns)),features.columns,rotation=90); save('13_feature_correlation')\n    plt.figure(); features.return_1.hist(bins=70); plt.xlabel('Equity daily return'); save('14_return_distribution')\n    plt.figure(); features.filter(like='vol_').drop(columns=['vol_ratio'],errors='ignore').plot(ax=plt.gca()); save('15_realized_volatility')\n    plt.figure(); features.filter(like='corr_').plot(ax=plt.gca()); save('16_correlations')\n"}
for name, text in SOURCES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text)
sys.path.insert(0, str(PROJECT))


## 2. Data and experiment settings
For actual market research set `DEMO=False` and `CSV_PATH` to an attached Kaggle dataset with date, equity and bonds columns (optional gold, oil, dollar, vix). Adjust split dates before viewing test results. Price columns must be adjusted and complete. No future filling is performed.

In [ ]:
DEMO = True
CSV_PATH = "/kaggle/input/your-dataset/market.csv"
CONFIG = {'seed': 42, 'states': [2, 3, 4, 5, 6], 'train_end': '2018-12-31', 'validation_end': '2021-12-31', 'transaction_cost': 0.001, 'risk_weights': [0.8, 0.6, 0.3, 0.1], 'windows': [5, 20, 60]}
from src.data_loader import synthetic, load_csv
prices = synthetic() if DEMO else load_csv(CSV_PATH)
print("SYNTHETIC SOFTWARE DEMO" if DEMO else "HISTORICAL DATA")
print(prices.shape, prices.index.min(), prices.index.max())
display(prices.head())

## 3. Feature engineering
All windows end on the observation date. Returns already capture momentum. Scaling and PCA are learned only from training. Inspect distributions and correlations before drawing economic conclusions; the fitting pipeline itself does not use historical event names.

In [ ]:
from src.features import make_features
features = make_features(prices, CONFIG["windows"])
display(features.describe().T)

## 4. Model fitting, validation and backtest
K-Means and GMM provide baselines; the HMM models persistent hidden states. State count is selected on validation log likelihood within the HMM family. It is not chosen on test returns. The fixed training fit is carried forward through validation and test. Online filtering never uses future observations.

The full run below also performs window/training-length validation sensitivity and transaction-cost/allocation sensitivity. A signal at close t is executed at close t+1 and earns returns thereafter. This cell may take several minutes.

In [ ]:
from src.pipeline import run
OUT = PROJECT / "outputs"
metrics = run(prices, CONFIG, OUT, DEMO)
display(metrics)

## 5. Can unsupervised ML find structure?
Silhouette is descriptive. K-Means score is negative squared distance, whereas GMM/HMM scores are log likelihood: do not compare these across families.

In [ ]:
import pandas as pd
display(pd.read_csv(OUT / "results/model_comparison.csv"))

## 6. What did the model discover?
State numbers are arbitrary. Interpret these training-only profiles before naming any state. Volatility rank determines allocation weights; recovery/crisis labels are not imposed.

In [ ]:
display(pd.read_csv(OUT / "results/training_state_profiles.csv"))
from IPython.display import Image, display
for n in ["01_market_regimes", "02_timeline", "03_profiles", "06_pca", "13_feature_correlation", "14_return_distribution", "15_realized_volatility", "16_correlations"]:
    display(Image(filename=str(OUT / "figures" / (n+".png"))))

## 7. How do markets transition?
The transition matrix estimates latent state persistence. Argmax label runs have a different interpretation. First and last durations are censored. Confidence is an inferred state probability, not a crash probability.

In [ ]:
display(pd.read_csv(OUT / "results/transition_matrix.csv"))
for n in ["04_transition", "05_probabilities", "11_durations"]:
    display(Image(filename=str(OUT / "figures" / (n+".png"))))

## 8. Does it generalize, and can it help manage risk?
These portfolio series use held-out observations. Compare against both equity and fixed 60/40. Lower drawdown alone does not demonstrate successful timing: reducing average equity exposure can mechanically reduce risk. Inspect return and Sharpe too. Risk-free rate and cash yield are set to zero.

In [ ]:
display(metrics)
for n in ["07_drawdown", "08_equity", "09_sharpe", "10_volatility", "12_returns_by_state"]:
    display(Image(filename=str(OUT / "figures" / (n+".png"))))
display(pd.read_csv(OUT / "results/performance_by_state.csv"))

## 9. Robustness
Structural sensitivity is evaluated on validation; test cost/allocation variants are exploratory. Do not select the best test variant and call it independent evidence. Likelihoods across different feature dimensions/scales are not directly comparable.

In [ ]:
display(pd.read_csv(OUT / "results/window_training_sensitivity.csv"))
display(pd.read_csv(OUT / "results/cost_allocation_sensitivity.csv"))

## 10. Leakage & Backtest Integrity
Trailing features; training-only transforms and state ranking; validation-only state-count choice; forward filtering; frozen test model; next-close execution; drift-aware two-leg costs and initial purchases. Training plots are retrospective, since the training fit saw that entire period. Historical adjusted data and fixed asset choices still introduce revision/selection limitations.

## 11. What failed? Conclusions and future work
If using demo mode, the only defensible conclusion is that the implementation runs. In the verified synthetic run, the regime strategy reduced drawdown relative to benchmarks but had a worse Sharpe ratio; that is not proof of financial value. For historical runs, document poor metrics and unstable parameter results explicitly. No statistical significance is claimed. Future work: walk-forward refits, block-bootstrap intervals, multiple HMM starts and point-in-time data.

In [ ]:
print((OUT / "results/REPORT.md").read_text())
print("Outputs saved to", OUT)